# 《我的修仙獎勵是不是發錯了？》第 1 集：Colab 直接打 MiniMax API（image-01 首格 + H3 片段）

照 `e001-jobs.json`（從 `video.json` 與開拍鎖定檔產生）依序做：**定妝照 → 首格 → 片段**，先做小樣 s33–s35，站主看過才放量。每一次付費請求都寫一筆收據到 Drive 的 `receipts.jsonl`，正文的 SHA-256 等於鎖定檔的。

**開始前**
1. Colab 左側「🔑 Secrets」加 `MINIMAX_API_KEY`（platform.minimax.io 的 API key；中國站帳號把下面 `BASE` 改成 `https://api.minimaxi.com`）。
2. 執行「設定」那一格，選 `STAGE`：`refs`（定妝照 3 張）→ `pilot_frames`（小樣 3 張首格）→ `pilot_clips`（小樣 3 支）→ 看過再 `frames`（其餘首格）→ `clips`（其餘片段）。
3. 做完把 Drive 的 `xiuxian-jiangli-e001/` 整個資料夾拿回本機的 `<VIDEO_WORKDIR>/xiuxian-jiangli-e001/`，接 `README.md` 的「拿回產線」。

**價目**（platform.minimax.io，2026-10-03 讀，請在帳號上核對）：image-01 每張 US$0.0035；H3 2K 每秒 US$0.13、768P 每秒 US$0.08。本集 42 支片段 169 秒，一次約 US$22、上限（每支兩次）約 US$44；小樣 3 支 US$1.56。`MAX_USD` 是這本筆記本的總停損。


In [ ]:
#@title 設定
EPISODE = "xiuxian-jiangli-e001"
JOBS_URL = "https://raw.githubusercontent.com/x812033727/travel_scanner/main/docs/videos/series-plans/xiuxian-jiangli/production/colab/e001-jobs.json"  #@param {type:"string"}
STAGE = "refs"  #@param ["refs", "pilot_frames", "pilot_clips", "frames", "clips", "poll", "report"]
RESOLUTION = "2K"  #@param ["2K", "768P"]
MAX_USD = 50.0  #@param {type:"number"}
BASE = "https://api.minimax.io"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/xiuxian-jiangli"  #@param {type:"string"}
ONLY_SHOTS = ""  #@param {type:"string"}
# ONLY_SHOTS: 逗號分隔的鏡號（例如 "s24,s25"）只做這幾鏡；空字串照 STAGE 的範圍。
REDO = False  #@param {type:"boolean"}
# REDO=True 才會對已經有檔的鏡再買一個 take（受 take 上限 frame 3 / clip 2 限制）。


In [ ]:
#@title 掛 Drive、讀工作清單、金鑰
import os, json, time, base64, hashlib, pathlib, datetime, requests
from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)
API_KEY = userdata.get("MINIMAX_API_KEY")
assert API_KEY, "Secrets 裡沒有 MINIMAX_API_KEY"
WORK = pathlib.Path(DRIVE_ROOT) / EPISODE
for sub in ("refs", "first-frames", "clips", "plan"):
    (WORK / sub).mkdir(parents=True, exist_ok=True)
jobs_path = WORK / "plan" / "e001-jobs.json"
r = requests.get(JOBS_URL, timeout=60); r.raise_for_status()
jobs_path.write_bytes(r.content)
JOBS = json.loads(jobs_path.read_text("utf-8"))
RECEIPTS = WORK / "receipts.jsonl"
PRICE = JOBS["pricing_usd"]
print(JOBS["title"]); print("refs", len(JOBS["refs"]), "frames", len(JOBS["frames"]), "clips", len(JOBS["clips"]), "pilot", JOBS["pilot"], "stills", JOBS["stills"])
print("estimate", JOBS["estimate_usd"])


In [ ]:
#@title 共用：收據、帳、API
HEAD = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
def sha(b): return hashlib.sha256(b if isinstance(b, bytes) else b.encode("utf-8")).hexdigest()
def now(): return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
def receipt(**row):
    row["at"] = now()
    with RECEIPTS.open("a", encoding="utf-8") as f: f.write(json.dumps(row, ensure_ascii=False) + "\n")
def spent():
    total = 0.0
    if RECEIPTS.exists():
        for line in RECEIPTS.read_text("utf-8").splitlines():
            try: total += float(json.loads(line).get("usd") or 0)
            except Exception: pass
    return total
def guard(cost):
    s = spent()
    if s + cost > MAX_USD:
        raise SystemExit(f"停：已花 US${s:.2f}，再花 US${cost:.2f} 會超過 MAX_USD {MAX_USD}")
def data_url(path):
    b = pathlib.Path(path).read_bytes()
    kind = "image/png" if b[:8] == b"\x89PNG\r\n\x1a\n" else "image/jpeg"
    return f"data:{kind};base64," + base64.b64encode(b).decode()
def takes(folder, sid, ext):
    return sorted(p for p in (WORK / folder).glob(f"{sid}-t*{ext}"))
def next_take(folder, sid, ext, cap):
    have = takes(folder, sid, ext)
    if have and not REDO: return None, have[-1]
    if len(have) >= cap: return None, have[-1]
    return len(have) + 1, None
def only(ids):
    if not ONLY_SHOTS.strip(): return ids
    want = {s.strip() for s in ONLY_SHOTS.split(",") if s.strip()}
    return [i for i in ids if i in want]

def gen_image(job, out_path, subject_path=None):
    body = {"model": JOBS["api"]["image_model"], "prompt": job["prompt"], "aspect_ratio": job["aspect"],
            "response_format": "base64", "n": 1, "prompt_optimizer": False}
    assert len(body["prompt"]) < 1500, (job["id"], len(body["prompt"]))
    if subject_path:
        body["subject_reference"] = [{"type": "character", "image_file": data_url(subject_path)}]
    guard(PRICE["image_01_per_image"])
    resp = requests.post(f"{BASE}/v1/image_generation", headers=HEAD, json=body, timeout=300)
    payload = resp.json()
    br = payload.get("base_resp") or {}
    if resp.status_code != 200 or br.get("status_code", 0) != 0:
        receipt(kind="image", id=job["id"], ok=False, http=resp.status_code, base_resp=br, usd=0)
        raise RuntimeError(f"{job['id']}: image failed {resp.status_code} {br}")
    data = payload.get("data") or {}
    if data.get("image_base64"):
        raw = base64.b64decode(data["image_base64"][0])
    else:
        raw = requests.get(data["image_urls"][0], timeout=120).content
    out_path.write_bytes(raw)
    receipt(kind="image", id=job["id"], ok=True, file=str(out_path), file_sha256=sha(raw), prompt_sha256=sha(job["prompt"]),
            subject=str(subject_path) if subject_path else None, usd=PRICE["image_01_per_image"])
    return out_path

def submit_video(job, frame_path):
    per_s = PRICE["h3_2k_per_second"] if RESOLUTION == "2K" else PRICE["h3_768p_per_second"]
    cost = per_s * job["duration"]
    guard(cost)
    body = {"model": JOBS["api"]["video_model"], "resolution": RESOLUTION, "duration": int(job["duration"]),
            "content": [{"type": "text", "text": job["text"]},
                        {"type": "image_url", "image_url": {"url": data_url(frame_path)}, "role": "first_frame"}]}
    assert sha(job["text"]) == job["sha256"], f"{job['id']} 正文跟鎖定檔不一致"
    resp = requests.post(f"{BASE}{JOBS['api']['video_endpoint']}", headers=HEAD, json=body, timeout=300)
    payload = resp.json() if resp.content else {}
    task_id = payload.get("task_id")
    if resp.status_code != 200 or not task_id:
        receipt(kind="clip-submit", id=job["id"], ok=False, http=resp.status_code, body=payload, usd=0)
        raise RuntimeError(f"{job['id']}: submit failed {resp.status_code} {payload}")
    task_id = str(task_id)
    receipt(kind="clip-submit", id=job["id"], ok=True, task_id=task_id, frame=str(frame_path), frame_sha256=sha(frame_path.read_bytes()),
            prompt_sha256=job["sha256"], duration=job["duration"], resolution=RESOLUTION, usd=cost)
    return task_id

def pending_tasks():
    done = {}; subs = {}
    if RECEIPTS.exists():
        for line in RECEIPTS.read_text("utf-8").splitlines():
            row = json.loads(line)
            if row.get("kind") == "clip-submit" and row.get("ok"): subs[row["task_id"]] = row
            if row.get("kind") == "clip-done": done[row["task_id"]] = row
    return {t: r for t, r in subs.items() if t not in done}

def poll_once():
    left = pending_tasks()
    for task_id, sub in list(left.items()):
        url = f"{BASE}/v2/query/video_generation/{task_id}"
        resp = requests.get(url, headers=HEAD, timeout=60)
        payload = resp.json() if resp.content else {}
        task = payload.get("task") or {}
        status = task.get("status") or ""
        if status in ("queued", "running", ""):
            continue
        if status == "succeeded":
            vurl = (task.get("content") or {}).get("url")
            raw = requests.get(vurl, timeout=600).content
            n = len(takes("clips", sub["id"], ".mp4")) + 1
            out = WORK / "clips" / f"{sub['id']}-t{n}.mp4"
            out.write_bytes(raw)
            receipt(kind="clip-done", id=sub["id"], task_id=task_id, ok=True, file=str(out), file_sha256=sha(raw), usd=0)
            print("done", sub["id"], out.name, f"{len(raw)/1e6:.1f} MB")
        else:
            err = task.get("error") or {}
            receipt(kind="clip-done", id=sub["id"], task_id=task_id, ok=False, status=status, error=err, usd=0)
            print("FAILED", sub["id"], status, err, "（1026／1027 是內容被拒，2013 是參數錯；被拒先回報，不換字繞）")
    return pending_tasks()
print("ok; 已花 US$%.2f" % spent())


In [ ]:
#@title 執行 STAGE
from IPython.display import display, Image as IPImage, HTML
REF_OF = {r["character"]: r for r in JOBS["refs"]}
def ref_path(cid):
    have = takes("refs", f"ref-{cid}", ".png")
    return have[-1] if have else None

if STAGE == "refs":
    for job in JOBS["refs"]:
        n, have = next_take("refs", job["id"], ".png", 3)
        if n is None: print("skip", job["id"], have.name); continue
        out = gen_image(job, WORK / "refs" / f"{job['id']}-t{n}.png")
        print("ref", job["name"], out.name); display(IPImage(filename=str(out), width=240))
    print("看臉：不對就 REDO=True 重做那一張（ONLY_SHOTS 填 ref-<id>）；對了再做 pilot_frames")

elif STAGE in ("pilot_frames", "frames"):
    ids = JOBS["pilot"] if STAGE == "pilot_frames" else [f["id"] for f in JOBS["frames"] if f["id"] not in JOBS["pilot"]]
    for job in JOBS["frames"]:
        if job["id"] not in only(ids): continue
        n, have = next_take("first-frames", job["id"], ".png", JOBS["take_caps"]["frame"])
        if n is None: print("skip", job["id"], have.name); continue
        subj = ref_path(job["subject"]) if job.get("subject") else None
        if job.get("subject") and subj is None: raise SystemExit(f"{job['id']} 需要 {job['subject']} 的定妝照，先跑 refs")
        out = gen_image(job, WORK / "first-frames" / f"{job['id']}-t{n}.png", subj)
        print(job["id"], out.name, "subject:", job.get("subject")); display(IPImage(filename=str(out), width=480))
    print("逐張看：景別、誰在左誰在右、動作開始前那一格、臉像不像定妝照；不對的用 ONLY_SHOTS + REDO 重畫")

elif STAGE in ("pilot_clips", "clips"):
    ids = JOBS["pilot"] if STAGE == "pilot_clips" else [c["id"] for c in JOBS["clips"] if c["id"] not in JOBS["pilot"]]
    for job in JOBS["clips"]:
        if job["id"] not in only(ids): continue
        have_clips = takes("clips", job["id"], ".mp4")
        if have_clips and not REDO: print("skip", job["id"], have_clips[-1].name); continue
        if len(have_clips) >= JOBS["take_caps"]["clip"]: print("cap", job["id"]); continue
        frames_ = takes("first-frames", job["id"], ".png")
        if not frames_: raise SystemExit(f"{job['id']} 沒有首格，先跑 frames")
        tid = submit_video(job, frames_[-1])
        print("submitted", job["id"], tid, f"{job['duration']}s {RESOLUTION}")
    print("送出後把 STAGE 改成 poll 再跑；官方建議每 10 秒查一次")

elif STAGE == "poll":
    left = pending_tasks()
    while left:
        print(now(), "還在等", sorted(r["id"] for r in left.values()))
        time.sleep(15)
        left = poll_once()
    for p in sorted((WORK / "clips").glob("*.mp4"))[-6:]:
        display(HTML(f"<p>{p.name}</p><video width=480 controls src='data:video/mp4;base64,{base64.b64encode(p.read_bytes()).decode()}'></video>"))
    print("全部回來了；已花 US$%.2f" % spent())

elif STAGE == "report":
    rows = [json.loads(l) for l in RECEIPTS.read_text("utf-8").splitlines()] if RECEIPTS.exists() else []
    by = {}
    for r in rows: by.setdefault(r.get("kind"), []).append(r)
    for k, v in by.items(): print(k, len(v), "ok", sum(1 for r in v if r.get("ok")), "USD %.2f" % sum(float(r.get("usd") or 0) for r in v))
    print("總計 US$%.2f；收據：%s" % (spent(), RECEIPTS))
    print("首格", len(list((WORK/'first-frames').glob('*.png'))), "片段", len(list((WORK/'clips').glob('*.mp4'))), "定妝照", len(list((WORK/'refs').glob('*.png'))))
